# Building a RAG System — Part 1: Ingestion Pipeline

**Internal Ship Program · Tasks 2.1 → 2.4**

Welcome! This notebook teaches the **ingestion** half of a Retrieval-Augmented Generation (RAG)
system, hands-on, using the **LangChain ecosystem** and **local open-source models** (no API keys,
your data never leaves the machine).

We process a real document — the **CIS Controls v8** security guidelines (`data/`) — through four stages:

```
        ┌──────────┐    ┌──────────┐    ┌───────────┐    ┌──────────────┐
  PDF → │ 2.1 PARSE│ →  │2.2 CHUNK │ →  │2.3 EMBED  │ →  │2.4 STORE     │ → (later: retrieve → rerank → agent)
        │unstructured   │splitters │    │bge-small  │    │Weaviate      │
        └──────────┘    └──────────┘    └───────────┘    └──────────────┘
```

| Stage | What it does | Tool |
|-------|--------------|------|
| **2.1 Parse** | Turn a messy PDF into clean text + structure (titles, tables, images) | `langchain-unstructured` |
| **2.2 Chunk** | Split text into retrieval-sized pieces | `langchain-text-splitters` + unstructured |
| **2.3 Embed** | Turn each chunk into a vector (numbers that capture meaning) | `langchain-huggingface` (`bge-small-en-v1.5`) |
| **2.4 Store** | Save vectors in a database we can search by similarity | `langchain-weaviate` |

> **Why RAG?** LLMs don't know your private docs. RAG lets the model *retrieve* relevant chunks
> from your documents at question time and answer grounded in them. Everything here is the
> "prepare the knowledge" half — retrieval, reranking, and the agent come next"


## 0. Setup

### Python dependencies
Already declared in `pyproject.toml`. If you cloned fresh, run **`uv sync`** in a terminal, then
select this project's `.venv` as the notebook kernel. (Or run the cell below to install on the fly.)

### System packages (required for hi-res PDF parsing)
`unstructured` uses OCR + a layout model to detect tables and images. On Debian/Ubuntu/WSL:

```bash
sudo apt-get update && sudo apt-get install -y poppler-utils tesseract-ocr libgl1
```

- **poppler-utils** → renders PDF pages to images
- **tesseract-ocr** → reads text from those images (OCR)
- **libgl1** → shared library the layout/vision model needs

> First run downloads the embedding model (~130 MB) and the layout model. Be patient once; they're cached after.


In [2]:
# 2.1 Parsing

# Import the Unstructured loader from LangChain.
# This loader uses the Unstructured library to extract content from PDFs
# while preserving document structure (titles, paragraphs, lists, tables, etc.).
from langchain_unstructured import UnstructuredLoader
# Utility function that removes metadata fields that cannot be serialized
# or stored easily in vector databases.
from langchain_community.vectorstores.utils import filter_complex_metadata
# Path provides operating-system-independent file paths.
# Using Path is more robust than hardcoding strings.
from pathlib import Path

# Construct the path to the PDF file.
# Path("..") moves one directory up from the current notebook/script location.
# This makes the code portable across different machines.
pdf_path = Path("..") / "data" / "CIS_Controls__v8__Critical_Security_Controls__2023_08.pdf"

# Create a loader for the PDF.
# strategy="hi_res" was chosen because:
# - It performs layout-aware parsing.
# - It attempts to preserve document structure.
# - It can better distinguish titles, headings, paragraphs, lists, and tables.
# - This often produces higher-quality chunks for Retrieval-Augmented Generation (RAG)
#   compared to simple text extraction.
loader = UnstructuredLoader(
    file_path=pdf_path,
    strategy="hi_res"
)

# Parse the PDF and extract document elements.
# Each element becomes a LangChain Document object containing:
# - page_content -> the extracted text
# - metadata -> information about the element
# Instead of returning one giant block of text, Unstructured splits the PDF
# into logical sections, which helps later chunking and retrieval.
documents = loader.load()

# Clean metadata.
# hi_res parsing generates a large amount of metadata such as:
# - coordinates
# - bounding boxes
# - layout information
# - page geometry
# Most vector databases do not need this information and some metadata
# structures cannot be serialized correctly.
# filter_complex_metadata removes these problematic fields while keeping
# useful metadata such as page numbers and source information.
documents = filter_complex_metadata(documents)

# Remove empty elements.
# Some PDFs produce blank elements during parsing.
# Examples:
# - empty lines
# - whitespace-only sections
# - parsing artifacts
# Keeping them would:
# - waste embedding computation
# - create useless vector entries
# - reduce retrieval quality
# Therefore we keep only elements that contain actual text.
documents = [
    d for d in documents
    if d.page_content and d.page_content.strip()
]

# Display how many usable document elements were extracted.
# This acts as a sanity check to verify parsing succeeded.
print(f"Parsed {len(documents)} elements")

# Preview a few parsed elements.
# Looking at the output helps verify:
# - the PDF was parsed correctly
# - headings are preserved
# - text is readable
# - no major extraction issues occurred
# This is an important debugging step before chunking and embedding.
print("\nFirst 3 elements preview:\n")

# Show the first three extracted elements.
# Only the first 300 characters are displayed so the output remains readable.
for i, doc in enumerate(documents[:3]):
    print(f"--- Element {i+1} ---")
    print(doc.page_content[:300])
    print()

C:\Users\Moe\AppData\Local\Temp\ipykernel_24416\2437106516.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores.utils import filter_complex_metadata
INFO: pikepdf C++ to Python logger bridge initialized
INFO: HTTP Request: HEAD https://huggingface.co/unstructuredio/yolo_x_layout/resolve/main/yolox_l0.05.onnx "HTTP/1.1 302 Found"
INFO: Reading PDF for file: ..\data\CIS_Controls__v8__Critical_Security_Controls__2023_08.pdf ...


Parsed 2651 elements

First 3 elements preview:

--- Element 1 ---
CIS Critical

--- Element 2 ---
Security Controls®

--- Element 3 ---
Version 8v8



In [3]:
#2.2 chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(f"\nChunks before cleaning: {len(chunks)}")

# Remove duplicates + tiny chunks
seen = set()
clean_chunks = []

for chunk in chunks:

    text = chunk.page_content.strip()

    if len(text) < 100:
        continue

    if text in seen:
        continue

    seen.add(text)
    clean_chunks.append(chunk)

chunks = clean_chunks

print(f"Chunks after cleaning: {len(chunks)}")

# Chunk statistics
sizes = [len(c.page_content) for c in chunks]

print("\nChunk Statistics")
print("----------------")
print("Min:", min(sizes))
print("Max:", max(sizes))
print("Avg:", round(sum(sizes)/len(sizes), 2))


Chunks before cleaning: 2766
Chunks after cleaning: 520

Chunk Statistics
----------------
Min: 100
Max: 500
Avg: 302.03


In [4]:
#2.3 Embedding
from langchain_huggingface import HuggingFaceEmbeddings

emb = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

print("\nEmbedding model loaded")

INFO: No device provided, using cpu
INFO: HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO: Loading SentenceTransformer model from BAAI/bge-small-en-v1.5.
INFO: HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

INFO: HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO: HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO: HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO: HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO: HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/tokenizer_config.json "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Req


Embedding model loaded


In [5]:
#2.4 Vector DB Setup

import weaviate

from weaviate.classes.config import (
    Property,
    DataType,
    Configure
)

client = weaviate.connect_to_local()

print("\nConnected:", client.is_ready())

collection_name = "Document"

if not client.collections.exists(collection_name):

    client.collections.create(
        name=collection_name,
        properties=[
            Property(
                name="text",
                data_type=DataType.TEXT
            ),
            Property(
                name="source",
                data_type=DataType.TEXT
            ),
            Property(
                name="page",
                data_type=DataType.INT
            ),
        ],
        vectorizer_config=Configure.Vectorizer.none()
    )

collection = client.collections.get(collection_name)



INFO: HTTP Request: GET http://localhost:8080/v1/.well-known/openid-configuration "HTTP/1.1 404 Not Found"
INFO: HTTP Request: GET http://localhost:8080/v1/meta "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://pypi.org/pypi/weaviate-client/json "HTTP/1.1 200 OK"
INFO: HTTP Request: GET http://localhost:8080/v1/.well-known/ready "HTTP/1.1 200 OK"
INFO: HTTP Request: GET http://localhost:8080/v1/schema/Document "HTTP/1.1 200 OK"



Connected: True


In [6]:
#2.5 Ingestion

texts = [chunk.page_content for chunk in chunks]

print("\nGenerating embeddings...")

vectors = emb.embed_documents(texts)

print("Embeddings generated")

with collection.batch.dynamic() as batch:

    for chunk, vector in zip(chunks, vectors):

        batch.add_object(
            properties={
                "text": chunk.page_content,
                "source": pdf_path.name,
                "page": chunk.metadata.get(
                    "page_number",
                    -1
                )
            },
            vector=vector
        )

print(f"\nInserted {len(chunks)} chunks")


Generating embeddings...


INFO: HTTP Request: GET http://localhost:8080/v1/schema/Document "HTTP/1.1 200 OK"
INFO: HTTP Request: GET http://localhost:8080/v1/nodes "HTTP/1.1 200 OK"


Embeddings generated


INFO: HTTP Request: GET http://localhost:8080/v1/nodes "HTTP/1.1 200 OK"



Inserted 520 chunks


In [126]:
#2.6 Retrieval

query = input("Ask a question: ")

query_vector = emb.embed_query(query)

results = collection.query.near_vector(
    near_vector=query_vector,
    limit=20,
    return_properties=["text"]
)

docs = [
    type("Doc", (), {"page_content": r.properties["text"]})
    for r in results.objects
]


print(f"Retrieved {len(docs)} candidates")

Retrieved 20 candidates


In [127]:
#2.7 reranking

from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Reranker loaded")


def rerank(query, docs):

    pairs = [
        (query, d.page_content[:1000])
        for d in docs
    ]

    scores = reranker.predict(pairs)

    reranked = sorted(
        zip(docs, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return reranked

INFO: No device provided, using cpu
INFO: HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 404 Not Found"
INFO: No modules.json found for cross-encoder/ms-marco-MiniLM-L-6-v2, initializing a new CrossEncoder model.
INFO: HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L-6-v2 "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L6-v2 "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

INFO: HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/processor_config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO: HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO: HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO: HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-

Reranker loaded


In [128]:
#2.8 Result
from langsmith import traceable
reranked_results = rerank(query, docs)

for rank, (doc, score) in enumerate(reranked_results[:5], start=1):
    print(f"\nRank {rank}")
    print(f"Score: {score:.4f}")
    print(doc.page_content[:250])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Rank 1
Score: 9.8098
The CIS Critical Security Controls® (CIS Controls®) started as a simple grassroots activity to identify the most common and important real-world cyber-attacks that affect enterprises every day, translate that knowledge and experience into positive, c

Rank 2
Score: 9.8098
The CIS Critical Security Controls® (CIS Controls®) started as a simple grassroots activity to identify the most common and important real-world cyber-attacks that affect enterprises every day, translate that knowledge and experience into positive, c

Rank 3
Score: 7.4693
CIS would like to thank the many security experts who volunteer their time and talent to support the CIS Critical Security Controls® (CIS Controls®) and other CIS work. CIS products represent the effort of a veritable army of volunteers from across t

Rank 4
Score: 7.4693
CIS would like to thank the many security experts who volunteer their time and talent to support the CIS Critical Security Controls® (CIS Controls®) and other 

In [135]:
passages = [
    doc.page_content
    for doc, score in reranked_results[:5]
]


import requests

from langsmith import Client
import uuid

client = Client()

run_id = uuid.uuid4()

client.create_run(
    id=run_id,
    name="RAG Answer",
    run_type="chain",
    inputs={"query": query}
)

url = "http://localhost:11434/api/generate"

def query_ollama(passages, query):
    prompt = f"""
You are an assistant that analyzes multiple text passages.

You will be given 5 passages below. Use ONLY these passages to answer the question or complete the task. If information is missing, say "Not enough information in the passages."

Passages:
1. {passages[0]}

2. {passages[1]}

3. {passages[2]}

4. {passages[3]}

5. {passages[4]}

Query:
{query}

Answer:
"""

    payload = {
        "model": "llama3.2:1b",
        "prompt": prompt,
        "stream": False
    }

    response = requests.post(url, json=payload)
    return response.json()["response"]


response = query_ollama(passages, query)
client.update_run(
    run_id,
    outputs={"answer": response}
)

print(response)


requests.post(
    "http://127.0.0.1:8000/process-answer",
    json={
        "run_id": str(run_id),
        "query": query,
        "answer": response,
        "passages": passages
    }
)

#requests.post(
 #   "http://127.0.0.1:8000/process-answer",
  #  json={
        
   #     "query": query,
    #    "answer": response,
     #   "passages": passages
    #}
#)

Based on the passages provided, I can conclude that CIS Critical Security Controls (CIS Controls) refer to a set of guidelines or best practices for protecting computer systems from cyber-attacks. These controls are mentioned in several passages as being started by CIS (Cybersecurity Information Sharing and Coordination Center), which is an organization focused on improving cybersecurity awareness and sharing knowledge among organizations.

The passages do not explicitly state what the CIS Critical Security Controls are, but based on the context and purpose of these guidelines, it can be inferred that they cover various aspects of security, such as:

* Identifying common cyber-attacks affecting enterprises
* Providing constructive action for defenders to improve their security posture
* Sharing knowledge with a wider audience

However, without further information or clarification from CIS itself, I must note that "CIS Critical Security Controls" is not a widely recognized term in the c

<Response [200]>

In [37]:
#Manual testing with DeepEval to test if it works

from deepeval.models import OllamaModel
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval import evaluate

model = OllamaModel(model="llama3.2:1b")

test_case = LLMTestCase(
    input="Why is Account Management considered critical?",
    actual_output="""
Account Management is considered critical because it ensures that only authorized users have access to enterprise systems and data. It reduces the risk of unauthorized access by managing user lifecycle, permissions, and account security controls. It is a key part of enforcing identity and access control within an organization.
""",
retrieval_context=[
    "CIS Control 05 Account Management defines the management of accounts to ensure proper identity lifecycle...",
    "Why is this Control critical? Account Management ensures that only authorized users have access to enterprise assets...",
    "CIS Control 06 Access Control Management focuses on restricting access based on roles and responsibilities...",
    "Account logging and monitoring is a critical component of security operations...",
    "Control 05 helps prevent unauthorized access and reduces attack surface by managing user accounts properly..."
]
)

faithfulness = FaithfulnessMetric(threshold=0.7, model=model)
relevancy = AnswerRelevancyMetric(threshold=0.7, model=model)

evaluate([test_case], [faithfulness, relevancy])

✨ You're running DeepEval's latest Faithfulness Metric! (using llama3.2:1b (Ollama), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using llama3.2:1b (Ollama), strict=False, 
async_mode=True)...

Output()

INFO: in _a_execute_llm_test_cases


INFO: HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


...

INFO: HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:            Why is Account Management considered critical?                                         │
│  │     Actual Output:                                                                                           │
│  │                       Account Management is considered critical because it ensures that only authorized      │
│  │                       users have access to enterprise systems and data. It reduces the risk of               │
│  │                       unauthorized access by managing user lifecycle, permissions, and account security      │
│  │                       controls. It is a key part of enforcing identity and access control within an          │
│  │                       organization.                                                                          │
│  │                                                                                                              │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric           ┃ Score ┃ Threshold ┃ Reason                                                    │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Faithfulness     │ 0.67  │ 0.70      │ The score of 0.67 indicates a discrepancy between the     │
│              │                  │       │           │ expected retrieval context (Account Management) and       │
│              │                  │       │           │ actual output ('Account Management ensures that only      │
│              │                  │       │           │ authorized users have access to enterprise systems and    │
│              │                  │       │           │ data').                                                   │
│        FAIL  │ Answer Relevancy │ 0.67  │ 0.70      │ The score is 0.67 because The input provided does not     │
│              │                  │       │           │ contain any irrelevant statements that would negatively   │
│              │                  │       │           │ impact the answer relevancy score.                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                                ┃ Average Score                  ┃ Pass Rate             ┃ Total         │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━ │
│  Faithfulness                          │ 0.67                           │ 0.00%                 │ 1             │
│  Answer Relevancy                      │ 0.67                           │ 0.00%                 │ 1             │
╰─────────────────────────────────────────────────────────

⚠ WARNING: No hyperparameters logged.
» ]8;id=921421;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 5.48s | token cost: None)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=False, metrics_data=[MetricData(name='Faithfulness', threshold=0.7, success=False, score=0.6666666666666666, reason="The score of 0.67 indicates a discrepancy between the expected retrieval context (Account Management) and actual output ('Account Management ensures that only authorized users have access to enterprise systems and data').", strict_mode=False, evaluation_model='llama3.2:1b (Ollama)', error=None, evaluation_cost=0.0, verbose_logs='Truths (limit=None):\n[\n    "CIS Control 05 Account Management defines the management of accounts to ensure proper identity lifecycle.",\n    "Account Management ensures that only authorized users have access to enterprise assets.",\n    "Control 06 Access Control Management focuses on restricting access based on roles and responsibilities.",\n    "Account logging and monitoring is a critical component of security operations.",\n    "Control 05 helps prevent unauthorized acces

In [38]:
from deepeval import evaluate
from deepeval.dataset import EvaluationDataset
from deepeval.dataset.golden import Golden
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.models import OllamaModel
from deepeval.test_case import LLMTestCase

# 1. Evaluation LLM

eval_model = OllamaModel(model="llama3.2:1b")

# 2. Questions

questions = [
    "What are CIS Implementation Groups?",
    "Why is Account Management considered critical?",
    "What does Control 05 do?",
    "How do CIS Controls improve security?",
    "What is the purpose of Access Control Management?"
]

# 3. Your RAG pipeline

def rag_pipeline(query):

    # Embed query
    query_vector = emb.embed_query(query)

    # Retrieve top documents
    results = collection.query.near_vector(
        near_vector=query_vector,
        limit=5,
        return_properties=["text"]
    )

    docs = [
        r.properties["text"]
        for r in results.objects
    ]

    # Generate answer
    answer = query_ollama(docs, query)

    return answer, docs

# 4. Create Goldens

goldens = []

for q in questions:

    answer, context = rag_pipeline(q)

    goldens.append(
        Golden(
            input=q,

            # Replace with human-written answers if available
            expected_output=answer,

            context=context
        )
    )

# 5. Build Dataset

dataset = EvaluationDataset(goldens=goldens)

# Save dataset
dataset.save_as(
    file_type="json",
    directory="./eval_data"
)

print("Dataset saved.")

# 6. Inspect sample

sample = dataset.goldens[0]

print("\nQuestion:")
print(sample.input)

print("\nExpected Output:")
print(sample.expected_output)

print("\nContext:")
print(sample.context)

# 7. Convert to Test Cases

test_cases = []

for golden in dataset.goldens:

    answer, retrieved_context = rag_pipeline(
        golden.input
    )

    test_cases.append(
        LLMTestCase(
            input=golden.input,
            actual_output=answer,
            expected_output=golden.expected_output,
            retrieval_context=retrieved_context
        )
    )


# 8. Metrics


metrics = [
    FaithfulnessMetric(model=eval_model),
    AnswerRelevancyMetric(model=eval_model)
]

# 9. Run Evaluation

results = evaluate(
    test_cases=test_cases,
    metrics=metrics
)

print(results)

Evaluation dataset saved at ./eval_data\20260605_130735.json!
Dataset saved.

Question:
What are CIS Implementation Groups?

Expected Output:
According to Passages 2 and 3, CIS Implementation Groups (IGs) were created as a recommended new guidance starting with Version 7.1 for prioritizing implementation of CIS Controls.

Context:
['The Center for Internet Security, Inc. (CIS®) makes the connected world a safer place for people, businesses, and governments through our core competencies of collaboration and innovation. We are a community-driven nonprofit, responsible for the CIS Controls® and CIS Benchmarks™, globally recognized best practices for securing IT systems and data. We lead a global community of IT professionals to continuously evolve these standards and provide products and services to proactively safeguard', 'result, starting with Version 7.1, we created CIS Controls Implementation Groups (IGs) as our recommended new guidance to prioritize implementation.', 'these recommend

✨ You're running DeepEval's latest Faithfulness Metric! (using llama3.2:1b (Ollama), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using llama3.2:1b (Ollama), strict=False, 
async_mode=True)...

Output()

INFO: in _a_execute_llm_test_cases


...

C:\Program 
Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\asyncio\selector_events.p
y:879: ResourceWarning: unclosed transport <_SelectorSocketTransport fd=6088 read=idle write=<idle, bufsize=0>>
  _warn(f"unclosed transport {self!r}", ResourceWarning, source=self)
ResourceWarning: Enable tracemalloc to get the object allocation traceback

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_3 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_4 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                                ┃ Average Score                  ┃ Pass Rate             ┃ Total         │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━ │
│  Faithfulness                          │ 0.67                           │ 100.00%               │ 5             │
│  Answer Relevancy                      │ 0.62                           │ 100.00%               │ 5             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=921423;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 25.81s | token cost: None)
» Test Results (5 total tests):
   » Pass Rate: 100.0% | Passed: 5 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

test_results=[TestResult(name='test_case_1', success=True, metrics_data=[MetricData(name='Faithfulness', threshold=0.5, success=True, score=0.6666666666666666, reason='The score of 0.67 indicates that the actual output is inconsistent with the retrieval context, specifically because it fails to mention administrative or highly privileged accounts as a target for attackers.', strict_mode=False, evaluation_model='llama3.2:1b (Ollama)', error=None, evaluation_cost=0.0, verbose_logs='Truths (limit=None):\n[\n    "Account logging is a critical component of security operations.",\n    "Account management procedures and tools are covered in CIS Control 5.",\n    "CIS Control 6 focuses on managing access for users, ensuring strong authentication, and assigning roles to users.",\n    "Accounts should only have the minimal authorization needed for their role.",\n    "Developing consistent access rights for each role is a best practice.",\n    "Administrative or highly privileged accounts are a t

In [ ]:
import requests
from deepeval.dataset import EvaluationDataset
from deepeval.dataset.golden import Golden

# -----------------------------
# Ollama helper
# -----------------------------
def ollama_generate(prompt):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3.2:1b",
            "prompt": prompt,
            "stream": False
        }
    )

    return response.json()["response"]

# -----------------------------
# Generate Goldens
# -----------------------------
goldens = []

# Generate ONLY 5 questions
for chunk in chunks[:5]:

    context = chunk

    question_prompt = f"""
You are creating evaluation questions for a RAG system.

Generate ONE question that can be answered from the passage below.

Passage:
{context}

Return ONLY the question.
"""

    question = ollama_generate(question_prompt).strip()

    answer_prompt = f"""
Answer the question using ONLY the passage.

Passage:
{context}

Question:
{question}

Return ONLY the answer.
"""

    expected_answer = ollama_generate(answer_prompt).strip()

    goldens.append(
        Golden(
            input=question,
            expected_output=expected_answer,
            context=[context]
        )
    )

    print(f"Generated: {question}")

# -----------------------------
# Build Dataset
# -----------------------------
dataset = EvaluationDataset(
    goldens=goldens
)

# -----------------------------
# Save Dataset
# -----------------------------
dataset.save_as(
    file_type="json",
    directory="./eval_data"
)

print(f"\nGenerated {len(dataset.goldens)} goldens")

# -----------------------------
# Inspect Sample Golden
# -----------------------------
g = dataset.goldens[0]

print("\nGenerated Question:")
print(g.input)

print("\nReference Answer:")
print(g.expected_output)

print("\nSource Context:")
print(g.context)

# -----------------------------
# Show All Generated Questions
# -----------------------------
print("\n===== ALL GENERATED QUESTIONS =====\n")

for i, golden in enumerate(dataset.goldens, start=1):
    print(f"{i}. {golden.input}")

Generated: What is one way to acknowledge the contributions of security experts who support CIS Critical Security Controls?
Generated: What is the primary purpose of Control 01 Inventory and Control of Enterprise Assets?
Generated: Why is Control 11 Data Recovery critical? 

Answer: It provides assurance that data can be restored in case of loss or corruption.
Generated: What type of administrator account is responsible for managing aspects of a computer, domain, or enterprise information technology infrastructure?
Generated: What is a database?
Evaluation dataset saved at ./eval_data\20260604_133846.json!

Generated 5 goldens

Generated Question:
What is one way to acknowledge the contributions of security experts who support CIS Critical Security Controls?

Reference Answer:
One way to acknowledge the contributions of security experts is to provide a link to their work, such as the Creative Commons License or the website provided by CIS.

Source Context:
['CIS Critical  \nSecurity Co

In [39]:
import requests
from deepeval.dataset import EvaluationDataset
from deepeval.dataset.golden import Golden

def ollama_generate(prompt):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3.2:1b",
            "prompt": prompt,
            "stream": False
        }
    )       
    return response.json()["response"]


# ---------------------------------
# Generate questions from scratch
# ---------------------------------

subject = "CIS Controls v8 cybersecurity framework"

goldens = []

for i in range(5):

    question_prompt = f"""
You are creating evaluation questions for a RAG system.

Subject:
{subject}

Generate ONE realistic question a cybersecurity professional might ask.

The question should be answerable from a CIS Controls document.

Return ONLY the question.
"""

    question = ollama_generate(question_prompt).strip()

    answer_prompt = f"""
You are a cybersecurity assistant.

Answer the following question about CIS Controls.

Question:
{question}

Provide a concise reference answer.
"""

    expected_answer = ollama_generate(answer_prompt).strip()

    goldens.append(
        Golden(
            input=question,
            expected_output=expected_answer,
            context=[]
        )
    )

    print(f"Generated Question {i+1}:")
    print(question)
    print()


# ---------------------------------
# Build Dataset
# ---------------------------------

dataset = EvaluationDataset(
    goldens=goldens
)

dataset.save_as(
    file_type="json",
    directory="./eval_data"
)

print(f"\nGenerated {len(dataset.goldens)} goldens")


# ---------------------------------
# Inspect Sample
# ---------------------------------

g = dataset.goldens[0]

print("\nQUESTION:")
print(g.input)

print("\nEXPECTED ANSWER:")
print(g.expected_output)

Generated Question 1:
What is the purpose of Control 9.5.1 in the CIS Controls v8 cybersecurity framework?

Generated Question 2:
What is the purpose of Control 6.4.1, specifically the requirement to "Verify all remote access connections are secure?"

Generated Question 3:
"Based on CIS Control 12: 'Identify and Assess Network Traffic', what is the minimum action required to assess network traffic for potential vulnerabilities, per the framework?"

Generated Question 4:
What is the purpose of Control 7.5.3.2, titled "Implement a Least Privilege Access Control and Authentication Mechanism" in the CIS Controls v8?

Generated Question 5:
What is the purpose of CIS Control 7.11, which requires organizations to ensure that they are not using an unverified vulnerability scan tool for scanning their computer systems?

Evaluation dataset saved at ./eval_data\20260605_130901.json!

Generated 5 goldens

QUESTION:
What is the purpose of Control 9.5.1 in the CIS Controls v8 cybersecurity framework

In [ ]:
import os
import json

from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import (
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric
)
from deepeval.models import OllamaModel
from deepeval.dataset.golden import Golden



folder = "./eval_data"

files = [f for f in os.listdir(folder) if f.endswith(".json")]
latest_file = sorted(files)[-1]

dataset_path = os.path.join(folder, latest_file)

print("Loading:", dataset_path)

with open(dataset_path, "r", encoding="utf-8") as f:
    data = json.load(f)



goldens = []

for item in data:
    goldens.append(
        Golden(
            input=item["input"],
            expected_output=item["expected_output"],
            context=item.get("context", [])
        )
    )



eval_model = OllamaModel(model="llama3.2:1b")



test_cases = []

for golden in goldens:

    actual_output, retrieved_context = rag_pipeline(golden.input)

    test_cases.append(
        LLMTestCase(
            input=golden.input,
            actual_output=actual_output,
            expected_output=golden.expected_output,
            retrieval_context=retrieved_context
        )
    )



metrics = [
    FaithfulnessMetric(model=eval_model, threshold=0.7),
    AnswerRelevancyMetric(model=eval_model, threshold=0.7),
    ContextualPrecisionMetric(model=eval_model, threshold=0.6),
    ContextualRecallMetric(model=eval_model, threshold=0.6)
]



results = evaluate(
    test_cases=test_cases,
    metrics=metrics
)

print(results)

Loading: ./eval_data\20260605_130901.json


✨ You're running DeepEval's latest Faithfulness Metric! (using llama3.2:1b (Ollama), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using llama3.2:1b (Ollama), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using llama3.2:1b (Ollama), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using llama3.2:1b (Ollama), strict=False, 
async_mode=True)...

...

C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\asyncio\selector_events.py:879: ResourceWarning: unclosed transport <_SelectorSocketTransport fd=6824 read=idle write=<idle, bufsize=0>>
  _warn(f"unclosed transport {self!r}", ResourceWarning, source=self)
C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\asyncio\selector_events.py:879: ResourceWarning: unclosed transport <_SelectorSocketTransport fd=6632 read=idle write=<idle, bufsize=0>>
  _warn(f"unclosed transport {self!r}", ResourceWarning, source=self)
C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\asyncio\selector_events.py:879: ResourceWarning: unclosed transport <_SelectorSocketTransport fd=6836 read=idle write=<idle, bufsize=0>>
  _warn(f"unclosed transport {self!r}", ResourceWarning, source=self)
C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:              What is the purpose of Control 9.5.1 in the CIS Controls v8 cybersecurity            │
│  │                         framework?                                                                           │
│  │     Actual Output:      Based on the passages provided, there is not enough information in the passages      │
│  │                         to determine the purpose of Control 9.5.1 in the CIS Controls v8 cybersecurity       │
│  │                         framework.                                                                           │
│  │                                                                                                              │
│  │                         Passage 3 mentions that starting with Version 7.1, CIS introduced itself as a way    │
│  │                         to "compromise the data, instead of an elaborate network and system hacking          │
│  │                         sequence", which implies that it may involve modifying or compromising systems       │
│  │                         rather than implementing controls to prevent attacks. However, this is not           │
│  │                         explicitly stated for Control 9.5.1.                                                 │
│  │                                                                                                              │
│  │                         It's also worth noting that passage 2 refers to CIS Controls Industrial Control      │
│  │                         Systems Implementation Guide (V8), but the passages do not provide information on    │
│  │                         what specific control (Control) in the guide relates to or what its purpose might    │
│  │                         be.                                                                                  │
│  │     Expected Output:    In the CIS Controls v8 cybersecurity framework, Control 9.5.1 is intended to         │
│  │                         promote employee awareness and training on data privacy best practices. This         │
│  │                         control aims to educate employees about their role in maintaining confidentiality    │
│  │                         and adhering to organizational policies related to data protection.                  │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Faithfulness         │ 0.50  │ 0.70      │ The CIS Controls' early stages were marked by a       │
│              │                      │       │           │ lack of formal structure and organization, which is   │
│              │                      │       │           │ evident in the fact that they started as a simple     │
│              │                      │       │           │ grassroots activity.                                  │
│        FAIL  │ Answer Relevancy     │ 0.67  │ 0.70      │ The score can be higher because some irrelevant       │
│              │                      │       │           

⚠ WARNING: No hyperparameters logged.
» ]8;id=921425;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 86.27s | token cost: None)
» Test Results (5 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 5

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

test_results=[TestResult(name='test_case_0', success=False, metrics_data=[MetricData(name='Faithfulness', threshold=0.7, success=False, score=0.5, reason="The CIS Controls' early stages were marked by a lack of formal structure and organization, which is evident in the fact that they started as a simple grassroots activity.", strict_mode=False, evaluation_model='llama3.2:1b (Ollama)', error=None, evaluation_cost=0.0, verbose_logs='Truths (limit=None):\n[\n    "The CIS Controls started as a simple grassroots activity.",\n    "The original goals were modest—to help people and enterprises focus their attention and get started on the most important steps to defend themselves from cyber-attacks.",\n    "CIS Controls were ordered in sequence to focus an enterprise’s cybersecurity activities.",\n    "Cyber hygiene was one of the subsets of CIS Controls referred to as \'cyber hygiene\'.",\n    "Starting with Version 7.1, CIS itself was created to compromise data instead of an elaborate network